# Improve human transcripts with Qwen3-Omni

This notebook uses audio together with its existing human label, recovers detected hallucinations from that label, and normalizes the selected result. It loads the same YAML as the paired benchmark.

Use an existing environment with Curator's `audio_cuda12` and `vllm` extras and FFmpeg. The inference cell needs two GPUs per ASR actor. See [README.md](README.md) for setup and output semantics. Configuration inspection does not load model weights.


In [ ]:
import os
from pathlib import Path

# Locate the repository when opening this notebook from its own directory.
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "nemo_curator").is_dir())
os.chdir(repo_root)
print(repo_root)

## Configure labelled audio

Set the manifest below to a local/shared JSONL containing `audio_filepath` and a non-empty `text` for every utterance. Optional `source_lang` defaults to English. Audio and text must align to the same complete clip; the tutorial rejects clips longer than 30 seconds instead of splitting their audio away from the reference.

Use a fresh output file and workspace for each run. All paths must be accessible to every worker in a multi-node cluster.


In [ ]:
from omegaconf import OmegaConf

manifest_path = Path("/data/human_labels.jsonl")  # Replace with your labelled manifest.
output_path = Path("/data/improved_labels.jsonl")
workspace_dir = Path("/data/qwen_improvement_workspace")

cfg = OmegaConf.load("tutorials/audio/qwen_omni_reference_improvement/pipeline.yaml")
cfg.manifest_path = str(manifest_path)
cfg.output_path = str(output_path)
cfg.workspace_dir = str(workspace_dir)
cfg.backend = "ray_data"  # Or "xenna"; cfg.execution_mode selects "streaming" or "batch".

print(OmegaConf.to_yaml(cfg, resolve=True))

## Inspect the pipeline

The English and multilingual prompts both embed `{transcript}`. A detected hallucination selects the original label before regex cleanup and abbreviation joining. Original `text`, raw `primary_model_prediction`, selected `best_prediction`, and final `abbreviated_text` remain separate fields. Sound-event detection and a second ASR model are not used.


In [ ]:
from nemo_curator.config.run import create_pipeline_from_yaml

pipeline = create_pipeline_from_yaml(cfg, log_config=False)
print(pipeline.describe())

## Run inference

This cell starts/connects to Ray and loads the pinned Qwen3-Omni model. Keep the same model revision when comparing backends. The output writer replaces the destination, so this cell refuses to reuse an existing output path. Inspect any reported failure before using partial output.


In [ ]:
from nemo_curator.config.run import create_executor_from_yaml, create_ray_client_from_yaml

if not manifest_path.is_file():
    msg = f"Set manifest_path to your labelled JSONL: {manifest_path}"
    raise FileNotFoundError(msg)
if output_path.exists():
    msg = f"Choose a fresh output_path: {output_path}"
    raise FileExistsError(msg)

ray_client = create_ray_client_from_yaml(cfg)
ray_client.start()
try:
    pipeline.run(executor=create_executor_from_yaml(cfg))
finally:
    ray_client.stop()

## Inspect proposed revisions and recovery

Compare each revision with its audio. A small edit distance from the human label does not prove higher accuracy. Rows with a non-empty `_skipme` remain in the output and should be reviewed before training. Selection source `reference` means hallucination recovery; `ground_truth` means unsupported-language fallback.


In [ ]:
import json
from collections import Counter

with output_path.open(encoding="utf-8") as stream:
    rows = [json.loads(line) for line in stream if line.strip()]

print("Rows:", len(rows))
print("Selection sources:", Counter(row["best_prediction_source"] for row in rows))
print("Flagged:", sum(bool(row.get("_skipme")) for row in rows))
for row in rows[:5]:
    print(
        json.dumps(
            {
                key: row.get(key)
                for key in (
                    "audio_filepath",
                    cfg.reference_text_key,
                    "primary_model_prediction",
                    "best_prediction",
                    "best_prediction_source",
                    "abbreviated_text",
                    "_skipme",
                )
            },
            ensure_ascii=False,
            indent=2,
        )
    )

## Compare execution backends

The [benchmark script](../../../benchmarking/scripts/audio_qwen_omni_reference_improvement_benchmark.py) uses this same YAML for paired Xenna and Ray Data runs on staged LibriSpeech labels. Use identical model revisions, input rows, and hardware when comparing throughput. The benchmark checks output coverage and reports reference-recovery counts; it does not claim transcription-quality gains.
